# Ejercicios de Inferencia Bayesiana

**Estadística Bayesiana** — Alfredo Sánchez Alberca

In [ ]:
# Configuración inicial
library(tidyverse)
library(rstan)
library(bayesplot)
library(tidybayes)
library(knitr)
options(mc.cores = parallel::detectCores())
rstan_options(auto_write = TRUE)

## Ejercicio 1 — Intervalo de credibilidad

Los datos históricos del sistema de vigilancia epidemiológica sugieren que la incidencia de la tuberculosis en una población está entre 8 y 15 casos por cada 100.000 habitantes-año. Si en el último año se han registrado 48 nuevos casos en una población de 300.000 habitantes, ¿cuál es el intervalo de credibilidad del 95% para la incidencia de la tuberculosis en esa población?

> Pista: Calibra una distribución Gamma como distribución a priori para la tasa de incidencia $\lambda$ (casos por cada 100.000 habitantes-año) ajustándola a la información disponible. Usa la aplicación https://nube.aprendeconalf.es/shiny/distribuciones/ para probar distintas distribuciones Gamma.

In [ ]:
# Definir el modelo Stan Gamma-Poisson
modelo_tuberculosis <- "
data {
  ... ;  // número de casos por cada 100.000 habitantes-año
}
parameters {
  ... ; // incidencia por cada 100.000 habitantes-año
}
model {
  ... ;       // función de verosimilitud
  ... ;  // distribución a priori calibrada
}
"

tuberculosis_post <- stan(model_code = modelo_tuberculosis,
    data = list(y = ...),
    chains = 4, iter = 5000 * 2, seed = 123)

In [ ]:
# Intervalo de credibilidad del 95% con summary
summary(tuberculosis_post, pars = "lambda", probs = c(..., ...))$summary 

In [ ]:
# Intervalo de credibilidad del 95% con tidybayes
tuberculosis_post |>
  spread_draws(lambda) |>
  mean_qi(... , .width = ...) 

In [ ]:
# Dibujar el intervalo de credibilidad
mcmc_intervals(tuberculosis_post, pars = "...",
  prob = 0.5, prob_outer = ...,
  point_est = "mean") +
  labs(title = "Intervalo de credibilidad del 95% para la incidencia de la tuberculosis") 

## Ejercicio 2 — Contraste de hipótesis bayesiano

Con el modelo ajustado en el ejercicio anterior para la incidencia de la tuberculosis:

a. Realizar el contraste bayesiano para ver si la incidencia es mayor que 15 por cada 100.000 habitantes-año. Es decir, calcular $P(\lambda > 15 \mid y)$.
b. ¿Cómo plantearías el contraste para ver si la incidencia es exactamente 18? Usar el intervalo de credibilidad del 99% para tomar la decisión.

In [ ]:
# Apartado a
# P(lambda > 15 | y)
tuberculosis_post |>
  spread_draws(lambda) |>
  summarise(prob = ...) 

In [ ]:
# Apartado b
# Intervalo de credibilidad del 99% para decidir si lambda = 18 es compatible con los datos
tuberculosis_post |>
  spread_draws(lambda) |>
  mean_qi(... , .width = ...)

## Ejercicio 3 — Factor de Bayes

Con el modelo ajustado para la incidencia de la tuberculosis, calcular el factor de Bayes para la hipótesis de que la incidencia es mayor que 15 por cada 100.000 habitantes-año, es decir

$$
H_0: \lambda > 15 \qquad H_1: \lambda \leq 15
$$

Interpreta el resultado.

In [ ]:
# Factor de Bayes para H0: lambda > 15
# Odds a priori: P(H0) / P(H1) usando la distribución a priori Gamma(..., ...)
tuberculosis_post |>
  spread_draws(lambda) |>
  summarise(
    odds_prior = (1 - pgamma(15, ..., ...)) / pgamma(15, ..., ...),
    odds_post  = ... / ...,
    BF         = odds_post / odds_prior
  ) 

## Ejercicio 4 — Distribución predictiva posterior

Con la distribución a posteriori obtenida para la incidencia de la tuberculosis, dibujar la distribución predictiva posterior del número de casos de tuberculosis en una nueva muestra de 10.000 personas, y dar un resumen de esta distribución con la media, desviación típica e intervalo de credibilidad del 95%.

In [ ]:
# Simulación de la distribución predictiva posterior
# Para una nueva muestra de 10000 personas, la tasa de la Poisson es lambda / 10
# (ya que lambda está expresado por cada 100.000 habitantes)
tuberculosis_pred <- tuberculosis_post |>
  spread_draws(lambda) |>
  mutate(pred = ... ) |>
  select(lambda, pred)
head(tuberculosis_pred, 5)

In [ ]:
# Diagrama de barras de la distribución predictiva posterior
tuberculosis_pred |>
  ggplot(aes(x = ..., y = ...) +
  stat_count(fill = "steelblue", width = 0.2) +
  labs(
    title = "Distribución predictiva posterior del número de casos de tuberculosis en 10.000 personas",
    x = "Número de casos", y = "Frecuencia relativa"
  ) +
  theme_gray()

In [ ]:
# Resumen: media, desviación típica e intervalo de credibilidad del 95%
tuberculosis_pred |>
  summarise(
    media = ... ,
    sd    = ... ,
    li    = ... ,
    ls    = ...
  )